# MSAE Neuron Labeling Notebook

## Imports + paths

In [ ]:
import re
import json
import time
from pathlib import Path

import pandas as pd
import numpy as np

from anthropic import Anthropic
import os

os.environ["ANTHROPIC_API_KEY"] = "" # Enter your API key here

Set paths:

In [117]:
TOP500_PATH = "msae_top500_items_per_neuron_50N.txt"

ITEMS_METADATA_PATH = "items_metadata.jsonl"   

RESULTS_DIR = Path("llm_results_msae50/amazon_fashion_msae")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## Parse the “top 500 per neuron” txt

In [118]:
def load_top_items_per_neuron(txt_path: str) -> dict[int, list[int]]:
    """
    Returns: { neuron_id: [idx0, idx1, ...] }
    """
    neuron_to_items = {}
    pattern = re.compile(r"^N(\d+):\s*\[(.*)\]\s*$")

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            m = pattern.match(line)
            if not m:
                raise ValueError(f"Unrecognized line format: {line[:120]}")

            neuron_id = int(m.group(1))
            items_str = m.group(2).strip()

            # robust split
            if items_str == "":
                item_idxs = []
            else:
                item_idxs = [int(x.strip()) for x in items_str.split(",") if x.strip()]

            neuron_to_items[neuron_id] = item_idxs

    return neuron_to_items

neuron_to_items = load_top_items_per_neuron(TOP500_PATH)
sorted(neuron_to_items.keys()), len(neuron_to_items[0])

([0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49],
 500)

## Load item csv data 

In [119]:
def load_items_metadata(path: str) -> pd.DataFrame:
    return pd.read_json(path, lines=True)

items_df = load_items_metadata(ITEMS_METADATA_PATH)
print(items_df.shape)
print(items_df.columns.tolist())
items_df = items_df.reset_index(drop=True)
items_df.head(2)

(220890, 16)
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,BALEAF Women's Long Sleeve Zip Beach Coverup U...,4.2,422,"[90% Polyester, 10% Spandex, Zipper closure, M...",[],31.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Women's UPF 50+ Front Zip Beach Co...,BALEAF,"[Clothing, Shoes & Jewelry, Women, Clothing, S...","{'Department': 'womens', 'Date First Available...",B09X1MRDN6,NaN,NaN,NaN
1,AMAZON FASHION,"SAS Women's, Relaxed Sandal",4.7,618,"[Made in the USA, Suede sole, Heel measures ap...","[Unwind, leave your worries behind, and simply...",188.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],SAS,"[Clothing, Shoes & Jewelry, Women, Shoes, Sand...",{'Product Dimensions': '10 x 15 x 6 inches; 2 ...,B0944VG4Y4,NaN,NaN,NaN


## Extraction of metadata per item

Format all metadata except "bought_together" as it can bias the label into co-purchase instead of concept

In [121]:
def safe_get(row: pd.Series, col: str, default=None):
    if col not in row:
        return default
    val = row[col]

    # treat None / NaN as missing, but allow lists/dicts/strings
    if val is None:
        return default
    if isinstance(val, float) and np.isnan(val):
        return default

    return val

def clip_str(s, max_len=200):
    s = "" if s is None else str(s).strip()
    return s if len(s) <= max_len else s[:max_len] + "…"

def as_list(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, (list, tuple)):
        return list(x)
    # sometimes single string
    return [x]

def format_features(features, max_items=6, item_max_len=90):
    feats = []
    for f in as_list(features)[:max_items]:
        sf = clip_str(f, item_max_len)
        if sf:
            feats.append(sf)
    return feats

def format_description(description, max_chars=220):
    # description is usually a list (possibly empty)
    parts = [clip_str(x, 200) for x in as_list(description) if clip_str(x, 1)]
    text = " ".join(parts).strip()
    return clip_str(text, max_chars)

def format_categories(categories, max_len=300):
    # in your data it's a list of strings path-like
    cats = [clip_str(c, 60) for c in as_list(categories) if clip_str(c, 1)]
    path = " > ".join(cats).strip()
    return clip_str(path, max_len)

def format_details(details, max_kv=7, key_max=32, val_max=70):
    # details is usually dict
    if details is None or (isinstance(details, float) and np.isnan(details)):
        return ""
    if isinstance(details, dict):
        items = []
        for k, v in list(details.items())[:max_kv]:
            k = clip_str(k, key_max)
            v = clip_str(v, val_max)
            if k and v:
                items.append(f"{k}: {v}")
        return clip_str("; ".join(items), 420)
    # fallback
    return clip_str(details, 200)

def count_media(media_field):
    # images/videos are lists of dicts; sometimes empty
    return len(as_list(media_field))

def extract_department(details):
    """
    details is usually a dict. Return normalized department string if present.
    """
    if isinstance(details, dict):
        dept = details.get("Department", None)
        if dept is None:
            return ""
        return clip_str(dept, 40).lower()  # normalize
    return ""

def format_item_evidence(row: pd.Series) -> dict:
    details_raw = safe_get(row, "details", None)
    dept = extract_department(details_raw)
    return {
        "asin": clip_str(safe_get(row, "parent_asin", ""), 40),

        "title": clip_str(safe_get(row, "title", ""), 180),
        "subtitle": clip_str(safe_get(row, "subtitle", ""), 120),

        "store": clip_str(safe_get(row, "store", ""), 80),
        "author": clip_str(safe_get(row, "author", ""), 80),

        "main_category": clip_str(safe_get(row, "main_category", ""), 50),
        "categories_path": format_categories(safe_get(row, "categories", None)),

        "department": dept,

        "price": safe_get(row, "price", None),
        "average_rating": safe_get(row, "average_rating", None),
        "rating_number": safe_get(row, "rating_number", None),

        "features": format_features(safe_get(row, "features", None)),
        "description_snippet": format_description(safe_get(row, "description", None)),

        "details": format_details(safe_get(row, "details", None)),

        # keep counts only (avoid URLs in prompt)
        "image_count": count_media(safe_get(row, "images", None)),
        "video_count": count_media(safe_get(row, "videos", None)),
    }

In [122]:
def get_item_row_by_index(idx: int, df: pd.DataFrame):
    # assumes indices refer to the row position in items_df
    if 0 <= idx < len(df):
        return df.iloc[idx]
    return None

## Neuron prompt (top-N only)


In [123]:
def build_neuron_prompt_payload(
    neuron_id: int,
    item_indices: list[int],
    items_df: pd.DataFrame,
    top_n: int = 25
) -> dict:
    chosen = item_indices[:top_n]
    evidence = []

    missing = 0
    for idx in chosen:
        row = get_item_row_by_index(idx, items_df)
        if row is None:
            missing += 1
            continue
        evidence.append({
            "item_index": int(idx),
            **format_item_evidence(row)
        })

    return {
        "neuron_id": int(neuron_id),
        "top_n_requested": int(top_n),
        "missing_items_in_metadata": int(missing),
        "items": evidence
    }

neuron_payloads = [
    build_neuron_prompt_payload(nid, neuron_to_items[nid], items_df, top_n=25)
    for nid in sorted(neuron_to_items.keys())
]
neuron_payloads[0]["items"][0]

{'item_index': 21227,
 'asin': 'B09DTQXJDP',
 'title': "RockDove Men's Original Two-Tone Memory Foam Slipper",
 'subtitle': '',
 'store': 'RockDove',
 'author': '',
 'main_category': 'AMAZON FASHION',
 'categories_path': 'Clothing, Shoes & Jewelry > Men > Shoes > Slippers',
 'department': 'mens',
 'price': 23.99,
 'average_rating': 4.5,
 'rating_number': 125407,
 'features': ['95% Cotton, 5% Spandex',
  'Rubber sole',
  'Easy on/off clog style slipper with a secure heel collar.',
  'Waffle knit upper ventilates the interior, letting your foot breathe and stay sweat-free; …',
  'Memory foam insole conforms to the contours of your foot for pillow soft comfort; pamper y…',
  'Sturdy rubber sole lets you step outside the house to grab the mail or walk the dog withou…'],
 'description_snippet': '',
 'details': 'Package Dimensions: 13.78 x 6.69 x 4.65 inches; 15.52 Ounces; Item model number: M302; Department: mens; Date First Available: March 16, 2018; Manufacturer: RockDove',
 'image_count'

## Claude Opus 4.5 labeling

### Prompts (system + user)

System prompt with minimal metadata to include broad theme:

In [124]:
SYSTEM_PROMPT = """
You are given a list of items that are most strongly associated with a latent neuron
from a recommender system. Each item includes a title and includes minimal tags
such as a short category path.

Your task is to identify the MOST consistent taxonomy pattern in the list:
Provide a BROAD label (high-level theme), e.g. "Women's shoes", "Men's clothing", "Jewelry", "Kids apparel", "Accessories".

Only return "No clear concept" if you cannot find a stable broad theme at all.

Rules:
- broad_label: short high-level theme.
- refined_label: REQUIRED. If broad_label resembles a previous label, refined_label MUST add a specific subtype
  using category/title patterns (e.g., "Women's apparel" -> "Women's apparel — dresses/skirts").
- Avoid exact repeats of broad_label if possible; if unavoidable, keep broad_label and make refined_label different.
- Use ONLY the evidence provided (titles/categories).
- Do NOT hallucinate unseen details.
- Output JSON only.
""".strip()

In [ ]:
def format_user_prompt(payload: dict, used_labels: list[str]) -> str:
    lines = []
    for it in payload["items"]:
        title = it.get("title", "")
        catp = it.get("categories_path", "")

        # keep only a short tag (avoid long paths)
        cat_short = " > ".join(catp.split(" > ")[:8]) if catp else ""

        tag_parts = []
        if cat_short:
            tag_parts.append(f"cat={cat_short}")

        tags = f" ({', '.join(tag_parts)})" if tag_parts else ""
        lines.append(f"- {title}{tags}")

    items_block = "\n".join(lines)
    used_block = ", ".join(used_labels[-50:]) if used_labels else "None"

    return f"""
Neuron ID: {payload['neuron_id']}

Previously used broad labels (avoid exact repeats; if similar, refine with a subtype):
{used_block}

Top-activating items (highest activation first):
<<<
{items_block}
>>>

Return ONLY valid JSON:
{{
  "neuron_id": {payload['neuron_id']},
  "broad_label": "BROAD_LABEL or No clear concept",
  "refined_label": "Refine the broad label with a subtype to avoid repetition",
  "description": "1-2 sentences grounded in repeated title/category patterns",
  "confidence": "low|medium|high",
  "evidence": ["3-6 short repeated patterns you relied on"]
}}
""".strip()

### API call wrapper + saving

In [126]:
# pip install anthropic
client = Anthropic()

In [127]:
def extract_json(text: str) -> dict:
    # Claude usually returns pure JSON if instructed, but keep this for robustness
    text = text.strip()
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError("No JSON object found.")
    return json.loads(m.group(0))

In [128]:
def label_neuron_with_claude(payload: dict, used_labels: list[str], sleep_s: float = 0.6) -> dict:
    user_prompt = format_user_prompt(payload, used_labels)

    resp = client.messages.create(
        model="claude-opus-4-5-20251101",
        temperature=0.1,
        max_tokens=450,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}]
    )

    time.sleep(sleep_s)

    text = resp.content[0].text
    try:
        return extract_json(text)
    except Exception as e:
        return {
            "neuron_id": payload["neuron_id"],
            "broad_label": "PARSE_ERROR",
            "refined_label": "PARSE_ERROR",
            "description": text[:800],
            "confidence": "low",
            "evidence": [],
            "error": str(e),
        }

Run labeling for all neurons:

In [129]:
all_results = []
used_labels = []

for payload in neuron_payloads:
    nid = payload["neuron_id"]
    print(f"Labeling neuron {nid} (items provided={len(payload['items'])})...")

    result = label_neuron_with_claude(payload, used_labels=used_labels)
    bl = result.get("broad_label", "").strip().lower()
    if bl and bl != "no clear concept" and bl != "parse_error":
        used_labels.append(bl)
        
    out_path = RESULTS_DIR / f"neuron_{nid:03d}_label.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    all_results.append(result)

with open(RESULTS_DIR / "all_neuron_labels.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

all_results[:2]

Labeling neuron 0 (items provided=25)...
Labeling neuron 1 (items provided=25)...
Labeling neuron 2 (items provided=25)...
Labeling neuron 3 (items provided=25)...
Labeling neuron 4 (items provided=25)...
Labeling neuron 5 (items provided=25)...
Labeling neuron 6 (items provided=25)...
Labeling neuron 7 (items provided=25)...
Labeling neuron 8 (items provided=25)...
Labeling neuron 9 (items provided=25)...
Labeling neuron 10 (items provided=25)...
Labeling neuron 11 (items provided=25)...
Labeling neuron 12 (items provided=25)...
Labeling neuron 13 (items provided=25)...
Labeling neuron 14 (items provided=25)...
Labeling neuron 15 (items provided=25)...
Labeling neuron 16 (items provided=25)...
Labeling neuron 17 (items provided=25)...
Labeling neuron 18 (items provided=25)...
Labeling neuron 19 (items provided=25)...
Labeling neuron 20 (items provided=25)...
Labeling neuron 21 (items provided=25)...
Labeling neuron 22 (items provided=25)...
Labeling neuron 23 (items provided=25)...
La

[{'neuron_id': 0,
  'broad_label': 'Clothing, Shoes & Jewelry',
  'refined_label': 'Clothing, Shoes & Jewelry — mixed general assortment',
  'description': "This neuron activates broadly across the entire Clothing, Shoes & Jewelry domain with no clear subcategory focus, spanning women's tops, men's slippers, novelty t-shirts, jewelry, dresses, and accessories without a dominant pattern.",
  'confidence': 'low',
  'evidence': ["Women's clothing (tops, dresses, sweaters)",
   "Men's items (slippers, jackets, belts)",
   'Jewelry (bracelets, necklaces, earrings)',
   'Novelty t-shirts',
   'Shoes (sneakers, sandals, boots)',
   'Baby pajamas']},
 {'neuron_id': 1,
  'broad_label': 'Clothing, Shoes & Jewelry',
  'refined_label': 'Clothing, Shoes & Jewelry — mixed family apparel & accessories',
  'description': "This neuron activates broadly across the entire Clothing, Shoes & Jewelry category with no specific demographic or product focus. Items span women's jewelry, men's activewear, girls'